# Graph RAG

In [1]:
import wikipediaapi
wiki_wiki = wikipediaapi.Wikipedia(user_agent='GraphRAG Project Reutlingen University (dominik.marc.neumann@icloud.com)', language='en')
page = wiki_wiki.page('Albert Einstein')

In [2]:
from llama_index.core import Document
documents = [
    Document(text=page.text)
]

In [3]:
import nest_asyncio
nest_asyncio.apply()

In [4]:
from llama_index.core import SimpleDirectoryReader, KnowledgeGraphIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.graph_stores import SimpleGraphStore
from llama_index.core import Settings

from llama_index.llms.ollama import Ollama

Settings.llm = Ollama(model="granite4:3b", request_timeout=200)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Chunking-Strategie
Settings.node_parser = SentenceSplitter(
    chunk_size=512,      # Tokens pro Chunk
)


In [6]:
index = KnowledgeGraphIndex.from_documents(
    documents,
    llm=Ollama(model="granite4:3b", request_timeout=200),
    embed_model=HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    show_progress=True,
)

Parsing nodes:   0%|          | 0/1 [00:00<?, ?it/s]

Processing nodes:   0%|          | 0/59 [00:00<?, ?it/s]

In [7]:
g = index.get_networkx_graph()

In [8]:
print(g)

Graph with 90 nodes and 88 edges


In [9]:
for node in g.nodes():
    print(node)

Alice
Bob
Philz
Coffee shop
Berkeley
1982
Switzerland
Aarau
Einstein
14 march 1879
18 april 1955
German-born
Swiss
American
United states
Germany
Us
1940
President franklin d. roosevelt
Manhattan project
Same paper as above
Quantum physics of radiation
Ulm
Kingdom of württemberg
German empire
St. peter's catholic elementary school
Luitpold gymnasium
Physics
Algebra
Calculus
Original proof of pythagorean theorem
Federal polytechnic school in zurich
Argovian cantonal school
German kingdom of württemberg
Federal polytechnic school
General part of test
Mathematics
Marcel grossmann
Mileva marić
Exploring science was more enjoyable than reading textbook solitude
September 1903
Marić
Misguided
Elsa löwenthal
14 february 1919
Löwenthal
Swiss patent office
Europe
1908-1933
University of bern
University of zurich
Full professorship
April 1911
Charles-ferdinand university
From october to november 1911
Eth zurich
Max planck
Walther nernst
Prussian academy of sciences
Apartment in dahlem
Director a

In [16]:
retriever = index.as_retriever(
    include_text=False,  # include source text, default True
)

nodes = retriever.retrieve("How many years did Einstein live in Swiss")

for node in nodes:
    print(node.text)

The following are knowledge sequence in max depth 2 in the form of directed graph like:
`subject -[predicate]->, object, <-[predicate_next_hop]-, object_next_hop ...`
['Einstein', 'Born on', '14 march 1879']
['Einstein', 'Died on', '18 april 1955']
['Einstein', 'Nationality', 'German-born']
['Einstein', 'Citizenship', 'Swiss']
['Einstein', 'Citizenship', 'American']
['Einstein', 'Was visiting in', 'United states']
['Einstein', 'Visited', 'Germany']
['Einstein', 'Decided to remain in', 'Us']
['Us', 'Supported by', 'Caltech']
['Einstein', 'Granted american citizenship', '1940']
['Einstein', 'Endorsed letter to', 'President franklin d. roosevelt']
['President franklin d. roosevelt', 'Alerted about', 'Potential german nuclear weapons program']
['Einstein', 'Recommended that us begin similar research', 'Manhattan project']
['Einstein', 'Demonstrated mass and energy equivalence from special theory', 'Same paper as above']
['Einstein', 'Worked with satyendra nath bose', 'Quantum physics of ra

In [17]:
query_engine = index.as_query_engine(
)

response = query_engine.query(
    "Tell me all about Einstein years in Swiss?"
)
print(response)

In Switzerland, Albert Einstein attended the Federal Polytechnic School in Zurich. He enrolled there at the age of seventeen and graduated with a mathematics and physics teaching diploma one year later. After graduating, he acquired Swiss citizenship and secured a permanent position at the Swiss Patent Office in Bern.
